In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

In [ ]:

#EXTRAE HASTA 100 RESULTADOS

query = "f1"
country = "ES"
leng = "ES"

def google_custom_search_df(query, country="ES", max_requests=10):
    API_KEY = os.getenv('API_KEY')
    API_CUSTOM_SEARCH_ID = os.getenv('API_CUSTOM_SEARCH_ID')

    if not API_KEY or not API_CUSTOM_SEARCH_ID:
        print("Error: API Key o ID de búsqueda no están cargados correctamente")
        return pd.DataFrame()

    search_results = []
    for i in range(1, max_requests * 10, 10):  # Limita a 10 resultados por búsqueda, hasta max_requests * 10
        url = f"https://www.googleapis.com/customsearch/v1?key={API_KEY}&cx={API_CUSTOM_SEARCH_ID}&q={query}&start={i}&gl={country}"
        response = requests.get(url)
        
        if response.status_code != 200:
            print(f"Error en la petición: {response.status_code}")
            break  # Detenemos las solicitudes si hay un error
        
        data = response.json()
        items = data.get("items", [])
        
        if not items:
            print("No se encontraron más resultados")
            break  # Detenemos si no hay más resultados
        
        search_results.extend(items)

    # Crear un DataFrame con los resultados de la búsqueda
    if search_results:
        df = pd.DataFrame(search_results, columns=['title', 'link'])
    else:
        df = pd.DataFrame()

    return df

# Llamada a la función con un límite de 10 requests (equivalente a 100 resultados)
df = google_custom_search_df(query, country, max_requests=10)
print(df)

In [ ]:
#EXTRAE SOLO 10 RESULTADOS

query = "publicar novela"
country = "ES"

def google_custom_search_df(query, country):
    API_KEY = os.getenv('API_KEY')
    API_CUSTOM_SEARCH_ID = os.getenv('API_CUSTOM_SEARCH_ID')

    if not API_KEY or not API_CUSTOM_SEARCH_ID:
        print("Error: API Key o ID de búsqueda no están cargados correctamente")
        return pd.DataFrame()

    # URL request para obtener solo los primeros 10 resultados
    url = f"https://www.googleapis.com/customsearch/v1?key={API_KEY}&cx={API_CUSTOM_SEARCH_ID}&q={query}&start=1&gl={country}"
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"Error en la petición: {response.status_code}")
        return pd.DataFrame()

    data = response.json()

    # Extraer los primeros 10 resultados
    items = data.get("items", [])

    if not items:
        print("No se encontraron resultados")
        return pd.DataFrame()

    # Crear un DataFrame con los resultados de la búsqueda
    df = pd.DataFrame(items, columns=['title', 'link'])

    return df

# Llamada a la función para obtener solo los primeros 10 resultados
df = google_custom_search_df(query, country)
print(df)

In [4]:
#-------------------SCRAPING del texto de los articulos de las 10 primeras posiciones de Google SERP

#Extraer el texto de los articulos con la libreria newspapper y, en su defecto, con beautiful soup
from bs4 import BeautifulSoup
from newspaper import Article

def scrape_article(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        if not article.text:
            # Si Newspaper no pudo obtener el texto, intenta con BeautifulSoup
            page = requests.get(url)
            soup = BeautifulSoup(page.content, 'html.parser')
            
            # Busca contenido en etiquetas <p> o <div>
            article_text = ' '.join([p.get_text() for p in soup.find_all(['p', 'div'])])
            
            return article_text
        return article.text
    except Exception as e:
        print(f"Error al scrapear el artículo: {str(e)}")
        return ""

def scrape_articles_in_dataframe(df):
    scraped_texts = []  # Aquí almacenaremos el texto de los artículos

    for url in df['link']:
        scraped_text = scrape_article(url)
        scraped_texts.append(scraped_text)

    # Agregamos los textos como una nueva columna en el DataFrame
    df['scraped_text'] = scraped_texts

    return df

# Llama a la función scraping de los artículos
df = scrape_articles_in_dataframe(df)

In [ ]:
#Configuraciones del LLM --- Ejecutar para definir temperatura y maximo de tokens
generation_config = {
    "temperature": 0.8,
    "top_p": 1,
    "top_k": 1,
    "max_output_tokens": 5000,
}

In [ ]:
#importar el modelo GEMINI con el que queremos trabajar
import google.generativeai as genai

# Configurar la API de Gemini (asumiendo que ya se ha configurado anteriormente)
genai.configure(api_key=os.environ['GEMINI_API_KEY'])

# Cargar el modelo Gemini 1.5 Flash
model = genai.GenerativeModel("gemini-1.5-flash") # ,generation_config=generation_config -- Avtivar para config temp y tokens)


In [ ]:
import time
# Función para hacer la llamada a Gemini 1.5 Flash y obtener un resumen y análisis SEO
def gemini_summarize_and_analyze(text):
    try:
        prompt = f"""You are an expert in semantic SEO optimization and content strategy.

        Summarize the following article and extract the most important points related to SEO, including keywords, bigraphs, trigrams, structure, meta tags and content strategy used.
        Focus on identifying strengths and weaknesses in SEO practices to determine what makes this article rank high in search engines.
        Keep in mind that all of this analysis will serve as the basis for creating new content that will outperform them in SEO.

        Artículo: {text}
        """
        
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        print(f"Error al generar el análisis con Gemini 1.5 Flash: {str(e)}")
        return "Error al generar análisis con Gemini 1.5 Flash"

# Función principal para procesar los artículos scrapeados
def process_scraped_articles(df):
    if df.empty:
        print("El DataFrame está vacío, no hay datos para procesar.")
        return df

    for index, row in df.iterrows():
        article_text = row['scraped_text']

        if not article_text.strip():  # Verifica si el texto está vacío
            print(f"Artículo vacío para la URL: {row['link']}, omitiendo análisis.")
            df.at[index, 'SEO Analysis'] = "No se pudo scrapear el texto del artículo"
            continue

        # Llamada a Gemini 1.5 Flash para analizar cada artículo
        print(f"Procesando artículo {index + 1}/{len(df)}: {row['link']}")
        summary_and_seo_analysis = gemini_summarize_and_analyze(article_text)
        df.at[index, 'SEO Analysis'] = summary_and_seo_analysis
        time.sleep(2)  # Agrega un pequeño retraso para no sobrecargar la API

    # Guardar el DataFrame a CSV
    output_csv = "Gemini_SEO_Analysis_on_SERP_Links_Text.csv"
    df.to_csv(output_csv, index=False)
    print(f"Archivo CSV generado: {output_csv}")

    return df

# Llama a la función para procesar los artículos scrapeados y generar el CSV
df = process_scraped_articles(df)

In [ ]:
def generar_guia_escritor_con_gemini(df):
    if 'SEO Analysis' not in df.columns:
        print("La columna 'SEO Analysis' no está presente en el DataFrame.")
        return

    # Concatenar todos los análisis SEO en un solo texto
    contexto_seo = "\n\n".join(df['SEO Analysis'].dropna().tolist())
    
    # Verificar el contenido de contexto_seo
    #print("Contenido de contexto_seo:", contexto_seo)  # Añadir esta línea para depuración

    # Prompt para Gemini
    prompt = f"""
    You are an expert at Semantic SEO. In particular, you are superhuman at taking the result of an SEO analysis of a search engine results page for a given keyword: {contexto_seo}.
    Using it to build a readout/guide that can be used to inform someone writing a long-form article about a given topic so that they can best fully cover the semantic SEO
    as shown in the SERP. The goal of this guide is to help the writer make sure that the content they are creating is as comprehensive to the semantic SEO
    expressed in the content that ranks on the first page of Google for the given {query}. With the following semantic data, please provide this readout/guide.
    This readout/guide should be useful to someone writing about the topic, and should not include instructions to add info to the article about the SERP itself.
    The SERP semantic SEO data is just to be used to help inform the guide/readout. Please provide the readout/guide in well organized and hierarchical markdown.
    Output Lenguage: spanish. Codification: UTF-8"
    """

    try:
        # Generar la guía con Gemini
        response = model.generate_content(prompt)
        guia_escritor = response.text

        print("Guía para el escritor generada con éxito.")
        return guia_escritor
    except Exception as e:
        print(f"Error al generar la guía con Gemini: {str(e)}")
        return "Error al generar la guía con Gemini"

# Llamar a la función para generar la guía
guia_para_escritor = generar_guia_escritor_con_gemini(df)

# Imprimir la guía generada
print("\nGuía para el escritor:")
print(guia_para_escritor)


In [ ]:
def generar_outline_con_gemini(guia_para_escritor, query):
    prompt = f"""
    Use the following writer's guide: {guia_para_escritor} and generate an incredibly thorough article outline.
    Consider all possible angles and be as thorough as possible. Do not add or modify the indications of the guide, 
    just develop each of the sections for our outline. Please provide the readout/guide in well organized and hierarchical markdown.
    Output Lenguage: spanish. Codification: UTF-8
    """
    try:
        response = model.generate_content(prompt)
        outline = response.text
        print("Esquema generado con éxito.")
        return outline
    except Exception as e:
        print(f"Error al generar el esquema con Gemini: {str(e)}")
        return "Error al generar el esquema con Gemini"

# Llamar a la función para generar el esquema
esquema_articulo = generar_outline_con_gemini(guia_para_escritor, query)

# Guardar el esquema en un archivo markdown
with open("esquema_articulo.md", "w", encoding="utf-8") as f:
    f.write(esquema_articulo)

print("\nEsquema del artículo guardado en 'esquema_articulo.md'")




In [ ]:
def generar_contenido_seccion(seccion, esquema_articulo):
    prompt = f"""
    Based on the outline provided, develop the content for the next section of the article.
    The content should develop only what defines the indicated section, should be informative and optimized for SEO.

    Be sure to follow these guidelines:

    1. **Introduction for the main section**: If it is a main heading, start with a brief introduction that summarizes the objective or relevance of the section.
    2. **Subsection development**: For each subsection, provide a detailed and complete development. Prioritize explanatory paragraphs over lists, except if the list is necessary for clarity.
    3. **Expansion of ideas**: Provide specific examples and details to optimize content for SEO, without conclusions or redundancies.

    Article outline:
    {esquema_articulo}

    Section to develop:
    {seccion}

    Output format: Markdown
    Output language: English
    Encoding: UTF-8

    """
    try:
        response = model.generate_content(prompt)
        contenido_seccion = response.text
        print(f"Contenido generado con éxito para la sección: {seccion[:50]}...")
        return contenido_seccion
    except Exception as e:
        print(f"Error al generar contenido para la sección {seccion[:50]}...: {str(e)}")
        return f"Error al generar contenido para la sección: {seccion[:50]}..."
def mejorar_contenido(contenido_seccion):
    prompt = f"""
    Actúa como editor y revisa el siguiente contenido expandiendo las subsecciones.
    
    Asegúrate de:
    1. Ampliar cada subsección con detalles adicionales, ejemplos y contexto donde corresponda.
    2. Convertir listas en párrafos descriptivos, excepto cuando una lista mejore la claridad del contenido.
    3. Revisar y optimizar el contenido para SEO, sin añadir conclusiones.
    4. Varía la estructura y el flujo del texto para:
        a. Diversificar el vocabulario y el estilo
        b. Incorpora expresiones idiomáticas y coloquiales
        c. Introducir errores tipográficos o gramaticales menores
        d. Añadir emociones, humor o sarcasmo (cuando sea apropiado)
        e. Incluir una voz personal
        f. Utilizar transiciones y conectores variados

    Contenido inicial:
    {contenido_seccion}

    Formato de salida: Markdown
    Idioma de salida: Español
    Codificación: UTF-8
    """

    try:
        response = model.generate_content(prompt)
        contenido_mejorado = response.text
        print("Contenido mejorado con éxito.")
        return contenido_mejorado
    except Exception as e:
        print(f"Error al mejorar contenido: {str(e)}")
        return f"Error al mejorar contenido."

def procesar_esquema(esquema_path, guia_para_escritor):
    contenido_mejorado = []
    seccion_actual = []

    with open(esquema_path, 'r', encoding='utf-8') as f:
        for linea in f:
            if linea.startswith("* **"):
                if seccion_actual:
                    seccion_texto = ''.join(seccion_actual).strip()
                    contenido_seccion = generar_contenido_seccion(seccion_texto, guia_para_escritor)
                    contenido_seccion = mejorar_contenido(contenido_seccion)
                    contenido_mejorado.append(contenido_seccion)
                    seccion_actual = []
                seccion_actual.append(linea)
            elif linea.startswith("**"):
                seccion_actual.append(linea)
            elif seccion_actual:
                seccion_actual.append(linea)

    if seccion_actual:
        seccion_texto = ''.join(seccion_actual).strip()
        contenido_seccion = generar_contenido_seccion(seccion_texto, guia_para_escritor)
        contenido_seccion = mejorar_contenido(contenido_seccion)
        contenido_mejorado.append(contenido_seccion)

    return contenido_mejorado

contenido_mejorado = procesar_esquema("esquema_articulo.md", guia_para_escritor)
contenido_completo = "\n\n".join(contenido_mejorado)

print("\nContenido completo generado:")
print(contenido_completo[:500] + "...")

with open("articulo_completo.md", "w", encoding="utf-8") as f:
    f.write(contenido_completo)

print("\nEl artículo completo ha sido guardado en 'articulo_completo.md'")


In [11]:
import base64
import requests
import os

def publicar_en_wordpress(titulo, archivo_contenido):
    # Obtener credenciales de WordPress desde variables de entorno
    login = os.getenv('WORDPRESS_LOGIN')
    password = os.getenv('WORDPRESS_PASSWORD')

    if not login or not password:
        print("Error: No se encontraron las credenciales de WordPress en las variables de entorno.")
        return

    # Configurar la URL de la API de WordPress y los encabezados de autorización
    url = 'https://archivofinal.com/wp-json/wp/v2/posts'
    headers = {
        'Authorization': 'Basic ' + base64.b64encode(f"{login}:{password}".encode()).decode()
    }

    # Leer el contenido del archivo
    try:
        with open(archivo_contenido, 'r', encoding='utf-8') as file:
            contenido = file.read()
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo {archivo_contenido}")
        return
    except IOError:
        print(f"Error: No se pudo leer el archivo {archivo_contenido}")
        return

    # Preparar el cuerpo de la solicitud
    data = {
        'title': titulo,
        'content': contenido,
        'status': 'draft'
    }

    # Realizar la solicitud POST
    try:
        response = requests.post(url, headers=headers, json=data)
        response.raise_for_status()
        print('Entrada creada correctamente en WordPress como borrador.')
    except requests.exceptions.RequestException as e:
        print(f'Error al crear la entrada en WordPress: {str(e)}')

# Llamar a la función para publicar en WordPress
publicar_en_wordpress("Publicar Novela", "articulo_completo.md")

Error al crear la entrada en WordPress: 500 Server Error: Internal Server Error for url: https://archivofinal.com/wp-json/wp/v2/posts
